# --- stage 1.1 (1/4): extraction of cc prices of 6 months ---

### using apis
##### for binance api, the limit is weight based, 500 candlestick has same weight for 1hr,4hr,...etc., 1hr will get a lot messy(fluctuating) data and more  long intervals will not get the trainable data hence, 4hr is a 'sweet spot' in between


In [ ]:
import requests
from bs4 import BeautifulSoup

# --- Getting the bitcoin data of last 6 Months interval: 4 hours ---
parameters={
    "symbol":"BTCUSDT",
    "interval":"4h",
    "limit": 500
}
req = requests.get(f"https://api.binance.com/api/v3/klines",params=parameters).json()
# get function is the GET request from https
# url takes the get function to the binance k-lines(candlestick) end-point
# params: instead of writing a long url; it straight away requests
# how it does it:
#   1. The params=params argument tells Python to:
#   2. Add a ? at the end of the base url.
#   3. Take every Key and Value from your dictionary.
#   4. Join them with an = sign.
#   5. Separate multiple pairs with an & symbol.
# the 'get' function will return the response from the server(ref. client-server arc response) along with raw data(wiz. json)
# .json will return the .json from the request

print(req)

# --- stage 1.2 (2/4): extraction of usd prices ---
### using library
- usually bitcoin is traded against dollar
- DXY is symbol of dollar index yahoo uses and nyb means New York Board of trade
- we are tracking DX-Y.NYB. This index measures the Dollar against six major world currencies (Euro, Yen, Pound, etc.).
- DXY < 100: The Dollar is weak. This is usually "Rocket Fuel" for Bitcoin.
- DXY > 105: The Dollar is very strong. This is a "Weight" that pulls the crypto market down.

In [ ]:
import yfinance

# --- Getting the data of USD for last 5 Days on intervals of 1 Hrs ---
ticker = yfinance.Ticker("DX-Y.NYB")
df = ticker.history(period="5d",interval="1h")
print(df)

# --- stage 1.3 (3/4): extraction of reddit sentiments ---
### Using JSON

In [1]:
import requests

ref="https://www.google.com/"
headers = {
        # Identity: Must reflect Windows and a recent Chrome version
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
        # Language: Tells the server the user's preferred language settings
        "Accept-Language": "en-US,en;q=0.9",

        # Connection Status: Standard practice for modern persistent connections
        "Connection": "keep-alive",

        # Previous website from which we referred this website
        "Referer": ref,

        "Upgrade-Insecure-Requests": "1",
        "Sec-Fetch-Dest": "document",
        "Sec-Fetch-Mode": "navigate",
        "Sec-Fetch-Site": "none",
        "Sec-Fetch-User": "?1",
        "Cache-Control": "max-age=0",
        "Accept-Encoding": "gzip, deflate, br",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,webp,image/apng,*/*;q=0.8",

}
# --- Getting .json response from reddit.json ---
re = requests.get("https://www.reddit.com/r/CryptoMarkets.json",headers=headers)

# --- Title Extraction from reddit posts
posts = re.json()['data']['children']
# data will take the values of the key 'data' from JSON dict
# children will return the list from the values : {'data':[d1,d2,d3]}
for p in posts:
    print(p['data']['title'])

[AMA] Polygon's Open Money Stack &amp; Acquisitions of Coinme and Sequence
Daily Crypto Discussion - January 29, 2026
Btc accumulation, and price expectation by 2030
Was there an altseason already?
Looking for a Spot Crypto Scalping Strategy/Indicator (Target: 0.1% Net Profit)
What crypto to invest in this year
Serenity sAxess Pro Is Officially Live
Barron Meme
Non KYC Exchange?
The Great Convergence: How Market Forces and Policy Are Enshrining Bitcoin in the American Economy. How the $1,000 Tax Exemption and 39% Merchant Adoption Are Turning Bitcoin into America's Everyday Currency.
Bottom litecoin.
Why will utility coin adoption result in higher coin prices?
Transferring BTC funds to ETF
Stop trading the noise. Zoom out. Most people look at Bitcoin through a microscope, panicking over every 10% dip. But when you look at the yearly lows, the signal becomes crystal clear. We aren't just looking at a price chart; we’re looking at the fastest-growing monetary network in history.
ERC-8004

# --- stage 1.4 (4/4): extraction of crypto news ---

In [4]:
from bs4 import BeautifulSoup
url = "https://www.coindesk.com/markets"
response = requests.get(url,headers=headers)
response.raise_for_status()
soup = BeautifulSoup(response.text,'html.parser')
heading_texts = [heading.text for heading in soup.find_all('h2')]
paras = soup.find_all('p')
texts=[]
for para in paras:
    texts.extend([i.get_text() for i in para.find_all('span')])

print(heading_texts,texts)

["Gold in 'extreme greed' sentiment as it adds entire bitcoin market cap in one day", 'Crypto Markets Today: Largest tokens decline, with derivatives signaling caution ahead', 'Investors accuse Cere Network of $100 million fraud and token-dump scheme in lawsuit', 'Metaplanet is raising $137 million to pay down debt and buy even more bitcoin', 'More than half of bitcoin investors are officially underwater as the $88,000 price level fails to hold', "Weaker dollar fails to spur bitcoin gains, but there's a reason for that", 'Number of wallets with 1 million XRP is rising again ', 'MegaETH mainnet to go live Feb. 9 in major test of ‘real-time’ Ethereum scaling', 'The Protocol: Ethereum to roll out new AI agents standard soon', 'Russia plans to cap retail crypto buys at $4,000 as it brings digital assets into the legal fold', 'SEC clarifies rules for tokenized stocks, tightening scrutiny on synthetic equity', 'Bitcoin trader warns of downside as gold rally continues to pull focus from BTC',